# PyTorch DDP Fashion MNIST Training Example

This example demonstrates how to train a convolutional neural network (CNN) to classify images using the [Fashion MNIST](https://github.com/zalandoresearch/fashion-mnist) dataset and [PyTorch Distributed Data Parallel (DDP)](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html).

This notebook walks you through running the example locally using the Kubeflow Trainer SDK, and how to scale it across multiple nodes on a Kubernetes cluster.

## Install Dependencies

Install the Kubeflow SDK and PyTorch dependencies. **After running this, please restart your kernel!**

In [35]:
!pip install -U kubeflow torch torchvision

## Define the Training Function

This function contains the core training logic. It handles the distributed setup, data loading, and the training loop.

In [36]:
def train_fn():
    # Standard imports inside the function to be self-contained
    import os
    import torch
    import torch.distributed as dist
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, DistributedSampler
    from torchvision import datasets, transforms

    # CNN Model Definition
    class Net(nn.Module):
        def __init__(self):
            super(Net, self).__init__()
            self.conv1 = nn.Conv2d(1, 20, 5, 1)
            self.conv2 = nn.Conv2d(20, 50, 5, 1)
            self.fc1 = nn.Linear(4 * 4 * 50, 500)
            self.fc2 = nn.Linear(500, 10)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            x = F.max_pool2d(x, 2, 2)
            x = F.relu(self.conv2(x))
            x = F.max_pool2d(x, 2, 2)
            x = x.view(-1, 4 * 4 * 50)
            x = F.relu(self.fc1(x))
            return self.fc2(x)

    # Distributed Setup
    world_size = int(os.environ.get("WORLD_SIZE", 1))
    rank = int(os.environ.get("RANK", 0))
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    
    if world_size > 1:
        # The Gloo backend is more stable for local process distribution
        backend = "gloo"
        dist.init_process_group(backend=backend)
        process_device = torch.device("cpu")
    else:
        process_device = torch.device("cpu")

    model = nn.parallel.DistributedDataParallel(Net().to(process_device)) if world_size > 1 else Net().to(process_device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.5)

    # Data Loading
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
    dataset = datasets.FashionMNIST("./data", train=True, download=True, transform=transform)
    
    # For quick testing, use a subset if requested
    if os.environ.get("KUBEFLOW_TRAINER_TEST") == "1":
        dataset = torch.utils.data.Subset(dataset, range(100))

    sampler = DistributedSampler(dataset) if world_size > 1 else None
    train_loader = DataLoader(dataset, batch_size=64, sampler=sampler, shuffle=(sampler is None))

    # Training Loop
    for epoch in range(1):
        model.train()
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(process_device), target.to(process_device)
            optimizer.zero_grad()
            output = model(data)
            loss = F.cross_entropy(output, target)
            loss.backward()
            optimizer.step()
            if batch_idx % 1 == 0 and rank == 0:
                percent = 100.0 * batch_idx / len(train_loader)
                print(f"Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} ({percent:.0f}%)] Loss: {loss.item():.6f}")

    if world_size > 1:
        dist.destroy_process_group()

## Option 1: Run Locally (Python Only)

The simplest way to verify the training logic works is to call the function directly in this notebook.

In [37]:
import os
os.environ["KUBEFLOW_TRAINER_TEST"] = "1"
train_fn()

100.0%
100.0%
100.0%
100.0%

Train Epoch: 0 [0/100 (0%)] Loss: 2.323055
Train Epoch: 0 [36/100 (50%)] Loss: 2.296972


## Option 2: Run Locally (using SDK)

This uses the SDK to run training in an isolated local process environment. This matches how it will behave on a cluster.

In [38]:
from kubeflow.trainer import TrainerClient, CustomTrainer, LocalProcessBackendConfig

# Configure local backend
backend_config = LocalProcessBackendConfig(cleanup_venv=True)
client = TrainerClient(backend_config=backend_config)

# Find the torch-distributed runtime
runtime = next(r for r in client.list_runtimes() if "torch" in r.name)

job_name = client.train(
    trainer=CustomTrainer(
        func=train_fn,
        packages_to_install=["torch", "torchvision"]
    ),
    runtime=runtime
)

print(f"Launched local job: {job_name}")

# Monitor local logs
for line in client.get_job_logs(job_name, follow=True):
    print(line, end="")

Launched local job: s48160969ea8
T h e   R P C   c a l l   c o n t a i n s   a   h a n d l e   t h a t   d i f f e r s   f r o m   t h e   d e c l a r e d   h a n d l e   t y p e .   
 
 E r r o r   c o d e :   B a s h / S e r v i c e / 0 x 8 0 0 7 0 7 2 c 
 
 [s48160969ea8-train] Completed with code 1 in 0:00:00.206711 seconds.T h e   R P C   c a l l   c o n t a i n s   a   h a n d l e   t h a t   d i f f e r s   f r o m   t h e   d e c l a r e d   h a n d l e   t y p e .     E r r o r   c o d e :   B a s h / S e r v i c e / 0 x 8 0 0 7 0 7 2 c   [s48160969ea8-train] Completed with code 1 in 0:00:00.206711 seconds.

## Option 3: Scale on Kubernetes Cluster

**Note**: Requires a working Kubernetes configuration (`~/.kube/config`).

In [39]:
from kubeflow.trainer import TrainerClient, CustomTrainer

try:
    client = TrainerClient()
    runtime = next(r for r in client.list_runtimes() if "torch" in r.name)
    
    job_name = client.train(
        trainer=CustomTrainer(
            func=train_fn,
            num_nodes=2,
            resources_per_node={"cpu": 2, "memory": "4Gi"},
            packages_to_install=["torch", "torchvision"]
        ),
        runtime=runtime
    )
    print(f"Launched cluster job: {job_name}")
    
    for line in client.get_job_logs(job_name, follow=True):
        print(line, end="")
except Exception as e:
    print(f"Could not connect to Kubernetes cluster: {e}")

Launched cluster job: nf7f75b18eb4
